In [1]:
import sys
import os
import copy
import shutil
import cv2
import matplotlib.pyplot as plt

import numpy as np
import torch

# Add the src directory to the path. TEMPORARY FIX
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.predictor import ShorelinePredictor

from src.data_processing.dataset_loader import CoastData

from src.models.metrics import Metrics

In [2]:
# Execute this cell to make sure 
# that external modules are reloaded
%load_ext autoreload
%autoreload 2

In [3]:
image_type_paths = {
    "oblique": {
        "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/processed_obliques_2_classes/")),
        "num_classes": 2,
        "weights_path": os.path.abspath(os.path.join(os.getcwd(), "../../artifacts/article/experiment4/oblique"))
    },
    "rectified": {
        "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/processed_rectified_3_classes/")),
        "num_classes": 3,
        "weights_path": os.path.abspath(os.path.join(os.getcwd(), "../../artifacts/article/experiment4/rectified"))
    }
}

networks: dict[str] = {
    "DeepLabV3": {
        "oblique": {
            "fold_0": "2026-01-08-14-11-42_oblique_fold_0_DeepLabV3_512x512",
            "fold_1": "2026-01-09-02-28-25_oblique_fold_1_DeepLabV3_512x512",
            "fold_2": "2026-01-09-16-02-54_oblique_fold_2_DeepLabV3_512x512",
            "fold_3": "2026-01-10-01-43-57_oblique_fold_3_DeepLabV3_512x512",
            "fold_4": "2026-01-10-19-43-05_oblique_fold_4_DeepLabV3_512x512"
        },
        "rectified": {
            "fold_0": "2026-01-11-12-39-10_rectified_fold_0_DeepLabV3_512x512",
            "fold_1": "2026-01-11-16-17-03_rectified_fold_1_DeepLabV3_512x512",
            "fold_2": "2026-01-11-19-26-44_rectified_fold_2_DeepLabV3_512x512",
            "fold_3": "2026-01-11-22-45-43_rectified_fold_3_DeepLabV3_512x512",
            "fold_4": "2026-01-12-01-55-44_rectified_fold_4_DeepLabV3_512x512"
        }
    }
}

patches = {
    "256x1024": {
        "patch_size": (256, 1024),
        "stride": (128, 512)
    },
}

In [4]:
for data_type in image_type_paths:
    print(f"\n{'#'*30}\nProcessing {data_type} images\n{'#'*30}")
        
    data_path = image_type_paths[data_type]["path"]
    num_classes = image_type_paths[data_type]["num_classes"]
    weights_path = image_type_paths[data_type]["weights_path"]

    print(f"Data path: {data_path}")

    # Load data
    data = CoastData(data_path)

    get_mask = True
    filtered_data = data.split_data(get_metadata=True, get_mask=get_mask)

    print(f"Number of samples: {len(filtered_data['test']['images'])}")
    for network in networks:
        # for patch_name, patch_info in patches.items():

        for fold in networks[network][data_type]:
            print(f"\n{'-'*20}\nPredicting with {network} - {data_type} - {fold}\n{'-'*20}")
            net_weights_path = os.path.join(weights_path, networks[network][data_type][fold], "models/best_model.pth")

            predictor = ShorelinePredictor(network, net_weights_path, num_classes)

            counter = 0
            total_images = len(filtered_data['test']['images'])

            ignore_index = 0 if data_type == "rectified" else None
            average = 'weighted' #  if data_type == "rectified" else 'macro'
            metrics = Metrics(
                phase='test',
                num_classes=num_classes,
                average=average,
                compute_loss=False,
                ignore_index=ignore_index
            )

            # Predict only the test set
            for path_img, path_mask, metadata in zip(filtered_data['test']['images'], filtered_data['test']['masks'], filtered_data['test']['metadata']):
                # Get filenames
                img_filename = os.path.basename(path_img)
                mask_filename = os.path.basename(path_mask)

                gt_mask = cv2.imread(path_mask, cv2.IMREAD_GRAYSCALE)

                # Predict
                landward_pixel_pred = 1 if data_type == "rectified" else 0
                seaward_pixel_pred = 2 if data_type == "rectified" else 1
                # print(patch_info["patch_size"], patch_info["stride"])
                output = predictor.predict(path_img, patch_size=patches["256x1024"]["patch_size"], stride=patches["256x1024"]["stride"], landward_pixel_pred=landward_pixel_pred, seaward_pixel_pred=seaward_pixel_pred)

                pred_mask = output['predicted_mask'].astype(np.uint8)
                
                # to tensor
                gt_mask_tensor = torch.tensor(gt_mask).unsqueeze(0)
                pred_mask_tensor = torch.tensor(pred_mask).unsqueeze(0)

                metrics.update_metrics(gt_mask_tensor, pred_mask_tensor)

            metrics.compute()
            print(metrics.get_last_epoch_info())



##############################
Processing oblique images
##############################
Data path: /home/josep/LOCALDATA/Shoreline-extraction/data/processed_obliques_2_classes
CoastData: global - 1717 images
Coast: agrelo, Total size: 244
Coast: arenaldentem, Total size: 40
Coast: cadiz, Total size: 946
Coast: cies, Total size: 430
Coast: samarador, Total size: 57
Number of samples: 174

--------------------
Predicting with DeepLabV3 - oblique - fold_0
--------------------
test metrics: 
	test_accuracy: 0.9552687406539917
	test_f1_score: 0.9552636742591858
	test_precision: 0.9552922248840332
	test_recall: 0.9552687406539917
	test_confusion_matrix: 
		0.9499 0.0501
		0.0396 0.9604


--------------------
Predicting with DeepLabV3 - oblique - fold_1
--------------------
test metrics: 
	test_accuracy: 0.9519177675247192
	test_f1_score: 0.9519060254096985
	test_precision: 0.9523069858551025
	test_recall: 0.9519177675247192
	test_confusion_matrix: 
		0.9371 0.0629
		0.0333 0.9667


--------

/home/josep/miniforge3/envs/shoreline/lib/python3.13/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: 3 NaN values found in confusion matrix have been replaced with zeros.
  warnings.warn(*args, **kwargs)


test metrics: 
	test_accuracy: 0.9746334552764893
	test_f1_score: 0.975053608417511
	test_precision: 0.9755148887634277
	test_recall: 0.9746334552764893
	test_confusion_matrix: 
		0.0000 0.0000 0.0000
		0.0011 0.9706 0.0283
		0.0006 0.0205 0.9788


--------------------
Predicting with DeepLabV3 - rectified - fold_1
--------------------
test metrics: 
	test_accuracy: 0.9758672714233398
	test_f1_score: 0.9762210249900818
	test_precision: 0.9766691327095032
	test_recall: 0.9758672714233398
	test_confusion_matrix: 
		0.0000 0.0000 0.0000
		0.0009 0.9697 0.0294
		0.0005 0.0171 0.9824


--------------------
Predicting with DeepLabV3 - rectified - fold_2
--------------------
test metrics: 
	test_accuracy: 0.9740877151489258
	test_f1_score: 0.9745122194290161
	test_precision: 0.9751151204109192
	test_recall: 0.9740877151489258
	test_confusion_matrix: 
		0.0000 0.0000 0.0000
		0.0009 0.9657 0.0334
		0.0008 0.0161 0.9831


--------------------
Predicting with DeepLabV3 - rectified - fold_3
-----